In [ ]:
# Import Pandas for working with CSV files
import pandas as pd

In [ ]:
# Open Tom's apparatus criticus CSV file and read it into a Pandas DataFrame
df = pd.read_csv("CSV/app-crit.csv")

In [ ]:
# Get basic information
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1185 entries, 0 to 1184
Data columns (total 67 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Unnamed: 0              1185 non-null   int64  
 1   Line                    1185 non-null   int64  
 2   Lemma                   1185 non-null   str    
 3   Lemma_Witnesses         200 non-null    str    
 4   Lemma_Sources           5 non-null      str    
 5   Lemma_Annotations       24 non-null     str    
 6   General_Comment         7 non-null      str    
 7   Reading_1               1178 non-null   str    
 8   Reading_1_Witnesses     1175 non-null   str    
 9   Reading_1_Sources       13 non-null     str    
 10  Reading_1_Annotations   99 non-null     str    
 11  Reading_2               310 non-null    str    
 12  Reading_2_Witnesses     310 non-null    str    
 13  Reading_2_Sources       1 non-null      str    
 14  Reading_2_Annotations   12 non-null     str    
 15

In [ ]:
# Inspect the first few rows of the DataFrame
df.head()

,Unnamed: 0,Line,Lemma,Lemma_Witnesses,Lemma_Sources,Lemma_Annotations,General_Comment,Reading_1,Reading_1_Witnesses,Reading_1_Sources,...,Reading_13_Sources,Reading_13_Annotations,Reading_14,Reading_14_Witnesses,Reading_14_Sources,Reading_14_Annotations,Reading_15,Reading_15_Witnesses,Reading_15_Sources,Reading_15_Annotations
0,1,1,ad,G P1 E T V H o,NaN,NaN,NaN,in,P F A Z H(spl),NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,1,bis iam,NaN,NaN,NaN,NaN,iam nunc,G,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,2,carmen,NaN,NaN,NaN,NaN,tempus,V,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,2,inerme,NaN,NaN,NaN,NaN,inane,o,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,5,nostri nisi me,NaN,NaN,NaN,NaN,nisi me nostri,A,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# Make the line number bold in Markdown
def process_line(line):
    bold_line = "**" + str(line) + "**"
    return bold_line

# Add an closing bracket to the lemma in Markdown
def process_lemma(lemma):
    lemma_plus_bracket = lemma + "]"
    return lemma_plus_bracket


# Helper function to handle witnesses and sources
def handle_witness(witness):
    if pd.isna(witness):
        return ""

    witness = str(witness)

    if len(witness) > 1 and witness[0].isalpha() and witness[1].isdigit():
        return f"*{witness[0]}^{witness[1:]}^*"

    elif (
        len(witness) > 1
        and witness[0].isalpha()
        and witness[1] == "("
        and witness[-1] == ")"
    ):
        return f"*{witness[0]}* (*{witness[2:-1]}*)"

    else:
        return f"*{witness}*"

# Process witnesses and sources for a reading
def process_witnesses(witnesses):
    if pd.isna(witnesses):
        return ""

    return " ".join(handle_witness(w) for w in str(witnesses).split())


# Process annotations for a reading
def process_annotations(annotations):
    if pd.isna(annotations):
        return ""

    return f"(*{annotations}*)"

# Process general comments for a reading
def process_general_comment(comment):
    if pd.isna(comment):
        return ""

    comment = str(comment)

    if comment.startswith("anne"):
        return f"*{comment[:4]}*{comment[5:]}"
    else:
        return f"*{comment[:2]}*{comment[3:]}"

# Process a single reading, including witnesses, sources, and annotations
def process_reading(row, i):
    parts = [
        str(row[f"Reading_{i}"]),
        process_witnesses(row[f"Reading_{i}_Witnesses"]),
        process_witnesses(row[f"Reading_{i}_Sources"]),
        process_annotations(row[f"Reading_{i}_Annotations"]),
    ]

    return " ".join(part for part in parts if part)

In [ ]:
# Loop over each row and process the entry.
for index, row in df.iterrows():
    # Initialize a list to hold the parts of the output for this row
    parts = []

    # Process the line number.
    parts.append(process_line(row["Line"]))
    # Process the lemma
    parts.append(process_lemma(row["Lemma"]))
    # Process the lemma's witnesses
    parts.append(process_witnesses(row["Lemma_Witnesses"]))
    # Process the lemma's sources
    parts.append(process_witnesses(row["Lemma_Sources"]))
    # Process the lemma's annotations
    parts.append(process_annotations(row["Lemma_Annotations"]))
    # Process the lemma's general comment
    parts.append(process_general_comment(row["General_Comment"]))
    # Add a colon and a space after the lemma and its associated information
    parts.append(": ")

    # Process each reading for this row
    for i in range(1, 15):
        if pd.notna(row[f"Reading_{i}"]):
            parts.append(process_reading(row, i))
            # If there are more readings, add a colon and a space after the reading
            if any(pd.notna(row[f"Reading_{j}"]) for j in range(i + 1, 15)):
                parts.append(": ")

    # Combine all the parts into a single string and assign it to the "output" column for this row
    df.at[index, "output"] = " ".join(part for part in parts if part)

In [ ]:
# Inpsect the output
df["output"]

0       **1** ad] *G* *P^1^* *E* *T* *V* *H* *o* :  in...
1                          **1** bis iam] :  iam nunc *G*
2                             **2** carmen] :  tempus *V*
3                              **2** inerme] :  inane *o*
4             **5** nostri nisi me] :  nisi me nostri *A*
                              ...                        
1180    **639** sint] *G* *P^1^* *E* *T* *H* :  sunt *...
1181                            **641** dent] :  tibi *G*
1182                  **641** rogatis] :  roganti *o* *ς*
1183                       **643** et] :  sed *H* *o* *ς*
1184                               **644** et] :  sub *G*
Name: output, Length: 1185, dtype: str

In [12]:
for output in df["output"]:
    print(output)

**1** ad] *G* *P^1^* *E* *T* *V* *H* *o* :  in *P* *F* *A* *Z* *H* (*spl*)
**1** bis iam] :  iam nunc *G*
**2** carmen] :  tempus *V*
**2** inerme] :  inane *o*
**5** nostri nisi me] :  nisi me nostri *A*
**5** me] :  nos *ς* *Heinsius*
**5** laesere] :  legere *A*
**7** unus] :  huius *G*
**8** non sinit] *an*noluit? : 
**9** nam] :  iam *Z*
**9** tacebo] *G* *P* *V* *H* *A* *o* :  latebit *P^1^* *E* *T* *F* *Z* *H* (*spl*)
**10** inassuetas] :  in aduersas *o*
**10** sumere] :  summere *A*
**11** relegatum] :  religatum *H*
**11** gelidas Aquilonis ad oras] *Eutyches* *V* *475.17* :  gelidos aquilonis ad ortus *ω*
**12** suo] *P* *E* *F* (*spl*) *o* *ς* *Eutyches* *V* *475.18* *Eutyches* *V* *475.18* :  meo *G* *P^1^* *T* *F* *V* *H* *Z* *A*
**13** immitis] :  inmittis *F* :  immerito *Z*
**14** nomina nostra] *G* *P* *V* *H* *Z* *o* :  uerba canina *P^1^* *E* *T* *F* *A* (*ex u. 232*) :  crimina nostra *ς*
**16** miseri] :  uiui *G* *ς* *edd.* *plerique*
**16** uiri] :  mei *P*
**17